# Database Fundamentals for AI — Portside Logistics / RouteMind

**Scenario:** Portside Logistics is a regional freight and parcel carrier running depots out of Chennai, Bengaluru, and Mumbai. Dispatchers and ops managers use **RouteMind**, an internal AI copilot, to ask questions about shipments, drivers, and company policy instead of digging through spreadsheets or PDFs by hand.

Before RouteMind can do anything clever with LangGraph or retrieval, it needs somewhere to keep its own operating data: every question a dispatcher asks it, how long each answer took, how many tokens it burned doing so, and which answers are worth caching. This notebook covers the practical database basics behind that layer:

- SQLite architecture and storage
- Schema design for AI workloads
- ORM vs raw SQL in SQLite
- File-backed vs in-memory connections
- Async database access with `aiosqlite`

SQLite is a serverless, embedded database engine that normally stores the complete database in a single file on disk, and it also supports in-memory databases for transient execution.

## Learning goals

By the end of this notebook, you should be able to:

1. Explain how SQLite stores data on disk.
2. Design tables for RouteMind's query history, token usage, latency, and cache records.
3. Choose between raw SQL and SQLModel for a local AI app.
4. Open file-backed and in-memory SQLite connections correctly.
5. Perform asynchronous CRUD operations with `aiosqlite`.

## 1) Install dependencies

Run this in a notebook cell:

```bash
pip install -U sqlmodel aiosqlite
```

You already get `sqlite3` with Python, so no extra install is needed for the standard library driver.

In [ ]:
%pip install -qU sqlmodel aiosqlite

## 2) SQLite architecture in one minute

SQLite is an in-process, serverless SQL database engine. The official docs describe it as a single-file database that normally stores the complete database in one disk file, with stable and well-defined file format details.

That makes SQLite a strong fit for RouteMind's own operating data:
- one file per depot deployment, easy to back up or copy
- no database server to run alongside the FastAPI process
- durable state for a small internal tool that a handful of dispatchers use at once

## 3) Why SQLite matters for AI workloads

RouteMind needs a simple persistence layer for:

- **query history** — every question a dispatcher or driver asked, and what RouteMind answered
- **response caching** — so a repeated question like "what's our SLA delay policy?" doesn't re-run the model
- **token usage logging** — per depot, so Portside can see which location is driving Groq costs
- **latency metrics** — how long RouteMind takes to answer, by model and request
- **trace checkpoints** — hooks for LangSmith once tracing is wired up
- **dispatcher/session state** — which depot and shift a query came from

SQLite is a good default here: low setup cost, a single portable database file, and more than enough throughput for a few dispatchers per depot.

## 4) Storage model

SQLite databases are usually stored in a single main database file. During a transaction, SQLite may also use a rollback journal or a WAL file depending on the journal mode. The architecture doc explains that tables and indexes are implemented with B-trees stored in the same disk file.

In [ ]:
from pathlib import Path

db_path = Path("routemind_fundamentals.db")
print("Database path:", db_path.resolve())

## 5) Schema design for AI workloads

A practical schema for RouteMind's own operating data includes a few tables:

- `routemind_query_log`: stores dispatcher questions and RouteMind's answers, tagged by depot
- `routemind_latency_log`: stores request timing and model metadata
- `routemind_token_usage`: stores input/output token counts, for cost tracking
- `routemind_cache_entries`: stores reusable answers or lookups

The goal is not to over-normalize too early. For a local AI app like this, simple transactional tables are often enough.

In [ ]:
import sqlite3
from datetime import datetime

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_query_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT NOT NULL,
    depot_city TEXT NOT NULL,
    role TEXT NOT NULL,
    prompt_text TEXT,
    response_text TEXT,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_latency_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_id TEXT NOT NULL,
    model_name TEXT NOT NULL,
    latency_ms REAL NOT NULL,
    status TEXT NOT NULL,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_token_usage (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    request_id TEXT NOT NULL,
    model_name TEXT NOT NULL,
    input_tokens INTEGER DEFAULT 0,
    output_tokens INTEGER DEFAULT 0,
    total_tokens INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS routemind_cache_entries (
    cache_key TEXT PRIMARY KEY,
    payload TEXT NOT NULL,
    model_name TEXT,
    expires_at TEXT,
    created_at TEXT NOT NULL
)
''')

conn.commit()
conn.close()

print("Tables created in", db_path)

## 6) Raw SQL vs ORM in SQLite

### Use raw SQL when:
- the query is simple
- you need full control
- you care about minimal abstraction
- you are doing one-off migration or debugging work

### Use SQLModel / SQLAlchemy when:
- the schema is shared across the app
- you want typed models
- you want cleaner CRUD code
- you are building a small-to-medium application with maintainable Python objects

SQLModel is built on Python type annotations and powered by Pydantic and SQLAlchemy. The official tutorial shows how to define table models and work with sessions step by step.

In [ ]:
from typing import Optional
from sqlmodel import SQLModel, Field, create_engine, Session, select

class RouteMindQueryLog(SQLModel, table=True):
    __tablename__ = "routemind_query_log_orm"
    id: Optional[int] = Field(default=None, primary_key=True)
    session_id: str
    depot_city: str
    role: str
    prompt_text: Optional[str] = None
    response_text: Optional[str] = None
    created_at: str

class RouteMindTokenUsage(SQLModel, table=True):
    __tablename__ = "routemind_token_usage_orm"
    id: Optional[int] = Field(default=None, primary_key=True)
    request_id: str
    model_name: str
    input_tokens: int = 0
    output_tokens: int = 0
    total_tokens: int = 0
    created_at: str

engine = create_engine(f"sqlite:///{db_path}", echo=False)
SQLModel.metadata.create_all(engine)

print("SQLModel tables created.")

In [ ]:
# Insert a sample row with SQLModel
now = datetime.utcnow().isoformat()

with Session(engine) as session:
    row = RouteMindQueryLog(
        session_id="chennai-depot-14",
        depot_city="Chennai",
        role="dispatcher",
        prompt_text="What's the ETA for shipment SHIP-1042 to Coimbatore?",
        response_text="SHIP-1042 departed the Chennai depot this morning and is on schedule for delivery today.",
        created_at=now,
    )
    session.add(row)
    session.commit()
    session.refresh(row)
    print("Inserted row:", row)

In [ ]:
# Read data back with SQLModel
with Session(engine) as session:
    statement = select(RouteMindQueryLog).where(RouteMindQueryLog.depot_city == "Chennai")
    rows = session.exec(statement).all()
    for r in rows:
        print(r)

## 7) When raw SQL is still the right choice

Raw SQL is often better for:
- fast inserts
- migrations
- schema inspection
- custom aggregations
- debugging query behavior

For RouteMind's logging tables, a raw SQL insert is often perfectly fine when the write path is simple and performance matters — every RouteMind response writes one latency row, and that write should stay cheap even under load.

In [ ]:
# Raw SQL example for logging RouteMind's response latency
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute('''
INSERT INTO routemind_latency_log (
    request_id, model_name, latency_ms, status, created_at
) VALUES (?, ?, ?, ?, ?)
''', ("req-8841", "openai/gpt-oss-20b", 842.5, "ok", datetime.utcnow().isoformat()))

conn.commit()

cur.execute("SELECT request_id, model_name, latency_ms, status FROM routemind_latency_log")
print(cur.fetchall())

conn.close()

## 8) File-backed vs in-memory connections

### File-backed database
Use a file path like:

```python
sqlite3.connect("routemind_fundamentals.db")
```

This is best for:
- persistent local storage
- repeatable development work
- storing state between runs

### In-memory database
Use:

```python
sqlite3.connect(":memory:")
```

This is best for:
- tests
- disposable experiments
- temporary data
- fast isolated demos

SQLite documents in-memory databases separately from disk-backed databases.

In [ ]:
# File-backed connection
file_conn = sqlite3.connect("routemind_fundamentals.db")
print("File-backed DB connected:", file_conn is not None)
file_conn.close()

# In-memory connection -- handy for a quick Chennai-depot smoke test
mem_conn = sqlite3.connect(":memory:")
mem_cur = mem_conn.cursor()
mem_cur.execute("CREATE TABLE demo (id INTEGER PRIMARY KEY, name TEXT)")
mem_cur.execute("INSERT INTO demo (name) VALUES (?)", ("temporary",))
mem_cur.execute("SELECT * FROM demo")
print("In-memory rows:", mem_cur.fetchall())
mem_conn.close()

## 9) Async database access with aiosqlite

`aiosqlite` provides an async interface to SQLite and mirrors the standard `sqlite3` API with async connection and cursor methods. It is useful once RouteMind moves behind FastAPI, where you do not want a blocking database call to stall the event loop while a dispatcher is waiting on an answer.

In [ ]:
import aiosqlite

async def init_async_db():
    async with aiosqlite.connect(db_path) as db:
        await db.execute("""
        CREATE TABLE IF NOT EXISTS routemind_async_cache (
            cache_key TEXT PRIMARY KEY,
            payload TEXT NOT NULL,
            created_at TEXT NOT NULL
        )
        """)
        await db.commit()

await init_async_db()

print("Async table created.")

In [ ]:
from datetime import datetime

async def async_create_read_update_delete():
    async with aiosqlite.connect(db_path) as db:

        # Create -- cache the answer to a common shipment-status question
        await db.execute(
            """
            INSERT OR REPLACE INTO routemind_async_cache
            (cache_key, payload, created_at)
            VALUES (?, ?, ?)
            """,
            (
                "shipment_status:SHIP-1042",
                "SHIP-1042 is on schedule, departed Chennai depot 2026-02-01 08:15",
                datetime.utcnow().isoformat(),
            ),
        )
        await db.commit()

        # Read
        async with db.execute(
            """
            SELECT cache_key, payload
            FROM routemind_async_cache
            WHERE cache_key = ?
            """,
            ("shipment_status:SHIP-1042",),
        ) as cursor:
            rows = await cursor.fetchall()
            print("Read:", rows)

        # Update -- shipment has now been delivered, refresh the cached answer
        await db.execute(
            """
            UPDATE routemind_async_cache
            SET payload = ?
            WHERE cache_key = ?
            """,
            (
                "SHIP-1042 was delivered to Coimbatore at 2026-02-01 17:40",
                "shipment_status:SHIP-1042",
            ),
        )
        await db.commit()

        # Verify update
        async with db.execute(
            """
            SELECT cache_key, payload
            FROM routemind_async_cache
            WHERE cache_key = ?
            """,
            ("shipment_status:SHIP-1042",),
        ) as cursor:
            print("After update:", await cursor.fetchall())

        # Delete -- cache entry expired
        await db.execute(
            """
            DELETE FROM routemind_async_cache
            WHERE cache_key = ?
            """,
            ("shipment_status:SHIP-1042",),
        )
        await db.commit()

        # Verify delete
        async with db.execute(
            """
            SELECT cache_key, payload
            FROM routemind_async_cache
            WHERE cache_key = ?
            """,
            ("shipment_status:SHIP-1042",),
        ) as cursor:
            print("After delete:", await cursor.fetchall())


await async_create_read_update_delete()

## 10) Choosing between sqlite3 and aiosqlite

### Use `sqlite3` when:
- your script is synchronous
- you want the simplest possible setup
- your data access happens in small batches

### Use `aiosqlite` when:
- your app is async already
- you want non-blocking DB access
- you are building an agent, API server, or event-driven workflow — like RouteMind sitting behind FastAPI

## 11) AI workload schema ideas

A simple but effective schema for RouteMind's own operating data:

- `routemind_query_log(session_id, depot_city, role, prompt_text, response_text, created_at)`
- `routemind_latency_log(request_id, model_name, latency_ms, status, created_at)`
- `routemind_token_usage(request_id, model_name, input_tokens, output_tokens, total_tokens, created_at)`
- `routemind_cache_entries(cache_key, payload, model_name, expires_at, created_at)`

This covers:
- auditability — what did a dispatcher ask, and what did RouteMind say back
- performance monitoring — is RouteMind getting slower at any one depot
- cost tracking — which depot's usage is driving the Groq bill
- cache reuse — don't re-answer the same policy question from scratch
- debugging — reconstruct exactly what happened for a support escalation

## 12) Practical recommendations

- Use a single SQLite file per depot deployment for local development and lightweight production.
- Use raw SQL for simple write paths and debugging, like the latency log.
- Use SQLModel when you want typed models and cleaner CRUD code, like the query log.
- Use `:memory:` for tests.
- Use `aiosqlite` once RouteMind is async, e.g. behind FastAPI.
- Keep the schema small, explicit, and focused on what RouteMind actually needs to log.

## 13) Mini exercise

Try this next:

1. Add a `shipment_id` column to `routemind_query_log` so a question can be linked directly to the shipment it was about.
2. Add a `response_time_ms` column to `routemind_query_log`.
3. Create an index on `depot_city` so a per-depot dashboard stays fast as the log grows.
4. Store a new row every time RouteMind answers a dispatcher.
5. Query the last 10 interactions for a given depot.

## Key takeaways

- SQLite is a single-file, serverless, embedded database engine.
- RouteMind's own operating data needs query history, token tracking, latency logs, and a cache table -- alongside whatever operational tables (shipments, drivers, customers) the business already has.
- Raw SQL is best for simple direct control, like a latency write.
- SQLModel is better when you want typed models and maintainable CRUD, like the query log.
- File-backed databases persist data; in-memory databases are temporary.
- `aiosqlite` gives you async, non-blocking SQLite access in Python.

## References

- SQLite architecture: https://sqlite.org/arch.html
- SQLite file format: https://sqlite.org/fileformat.html
- SQLite single-file database: https://www.sqlite.org/onefile.html
- SQLite documentation: https://sqlite.org/docs.html
- SQLite appropriate uses: https://sqlite.org/whentouse.html
- SQLModel tutorial: https://sqlmodel.tiangolo.com/tutorial/
- SQLModel homepage: https://sqlmodel.tiangolo.com/
- aiosqlite: https://github.com/omnilib/aiosqlite
- aiosqlite docs: https://aiosqlite.omnilib.dev/en/latest/
- Intro to SQLite in Python: https://www.geeksforgeeks.org/python/introduction-to-sqlite-in-python/
- SQLite tutorial: https://www.geeksforgeeks.org/sqlite/sqlite-tutorial/